# Born Effective Charges of Zincblende-Structure Crystals    
**by <span style="color:darkgreen">Sebastian Tillack</span> for [<span style="color:darkgoldenrod">exciting *sodium*</span>](https://www.exciting-code.org/sodium)**

<font size="2">(Jupyter notebook by <span style="color:darkgreen">Mara Voiculescu</span> & <span style="color:darkgreen">Martin Kuban</span>)</font>
<hr style="border:2px solid #DDD"> </hr>

**<span style="color:firebrick">Purpose</span>**: In this tutorial, you will learn how to compute **Born effective charges** from the macroscopic polarization in zincblende crystals.
<hr style="border:2px solid #DDD"> </hr>

<div class="alert alert-block alert-warning">

**Table of Contents**
    
[0. Before Starting](#0)
    
[1. Theoretical Background](#1)  
    
[2. Born Effective Charges Using Finite Differences](#2)   
    
[3. Born Effective Charges from DFPT](#3)      
    
   - [Exercises](#4)
    
</div>

<a id='0'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">0. Before Starting</span>

**<span style="color:firebrick">Read the following paragraphs before starting with the rest of this tutorial!</span>**

Before running any Jupyter tutorials, please refer to the **`00_before_starting.md`** document on how to correctly set up the environment. This only needs to be done once. After which, the **venv** can be (re)activated from **`exciting`**'s root directory:

<div style="background-color: rgb(224, 224, 224);">

```bash
source $EXCITINGROOT/tools/excitingjupyter/venv/excitingvenv/bin/activate
```

</div>

<a id='1'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">1. Theoretical Background</span>

The **Born effective charge** (or dynamical charge) ${\bf Z}_\alpha^\star$ of an atom $\alpha$ describes the effective dipole ${\bf p}$ that is induced in the unit cell due to a displacement ${\bf d}$ of the atom from its equilibrium position:

\begin{equation}
\tag{1}
{\bf p} = {\bf Z}_\alpha^\star \cdot {\bf d}\;.
\end{equation}

Note, that in general, ${\bf Z}_\alpha^\star$ is a 3×3 tensor and the dipole moment is not necessarily parallel to the displacement. The Born effective charge tensor is defined as the negative second derivative of the total energy with respect to an external electric field ${\bf \mathcal{E}}$ and the atomic position ${\bf \tau}_\alpha$ evaluated at the equilibrium position:

\begin{equation}
\tag{2}
Z^\star_{\alpha,ij} = - \left.\frac{\partial^2 E}{\partial \mathcal{E}_j\, \partial \tau_{\alpha,i}}\right|_0\;.
\end{equation}

In practice, we compute the Born effective charges from the **macroscopic polarization** ${\bf \mathcal{P}}$ that is given by

\begin{equation}
\tag{3}
\mathcal{P}_i = - \frac{1}{\Omega} \left.\frac{\partial E}{\partial \mathcal{E}_i}\right|_0\;,
\end{equation}

where $\Omega$ is the unit cell volume. With that, the charge tensor reads

\begin{equation}
\tag{4}
Z^\star_{\alpha,ij} = \left. \Omega \frac{\partial \mathcal{P}_j}{\partial \tau_{\alpha,i}}\right|_0\;.
\end{equation}

In order to evaluate the macroscopic polarization, we employ the so-called Berry phase approach.

Being defined via the derivative of the polarization with respect to nuclei displacements, Born charges are naturally closely connected to phonons. Therefore, we will use the <code><span style="color:green">phonons</span></code> element to compute them. See also the tutorial **Lattice Dynamics of Diamond and Zincblende-Structure Crystals**.

<a id='2'></a>
<hr style="border:1px solid #DDD"> </hr>

### <span style="color:#15317E">2. Born Effective Charges Using Finite Differences</span>

In order to compute Born charges using finite differences, we have to run a **supercell** phonon calculation at the **Γ**-point. First, create the new directories.

In [1]:
%%bash
mkdir -p run_cBN_Borncharges && cd run_cBN_Borncharges
mkdir -p supercell
cd ..

We need to displace each atom in the unit cell in each of the Cartesian directions and evaluate the macroscopic polarization. This procedure is very similar to a **Γ**-point phonon calculation. Therefore, the Born effective charges are implemented as a side product of a regular **Γ**-point phonon calculation. *I.e.*, in order to trigger a Born effective charge calculation, we need to include the <code><span style="color:green">phonons</span></code> element in the input file.

<span class="BN_BORNCHARGES"></span>
```xml
<input>
 
   <title>cBN Born effective charges - supercell</title>
 
   <structure speciespath="$EXCITINGROOT/species/">
      <crystal scale="6.850832">
         <basevect>0.0   0.5   0.5</basevect>
         <basevect>0.5   0.0   0.5</basevect>
         <basevect>0.5   0.5   0.0</basevect>
      </crystal>
      <species speciesfile="B.xml">
         <atom coord="0.00 0.00 0.00"/>
      </species>
      <species speciesfile="N.xml">
         <atom coord="0.25 0.25 0.25"/>
      </species>
   </structure>
 
   <groundstate 
      do="fromscratch"
      ngridk="4 4 4" 
      rgkmax="4.0"
      gmaxvr="14" 
      xctype="GGA_PBE" 
      swidth="0.0001" 
      />
 
   <phonons
      do="fromscratch"
      method="sc"
      ngridq="1 1 1"
      >
   </phonons>
 
</input>
```

In [2]:
from excitingjupyter.utilities import get_input_xml_from_notebook   

# Extract input file content from this notebook:
input_str = get_input_xml_from_notebook("born_effective_charges_of_zincblende_structure_crystals", "BN_BORNCHARGES")

# Write out the input as an XML file:
with open('./run_cBN_Borncharges/supercell/input.xml', "w") as fid:
    fid.write(input_str)

Note, that we set the attribute <code><span style="color:mediumblue">ngridq</span>=<span style="color:firebrick">"1 1 1"</span></code> which corresponds to a phonon calculation at **Γ** only.

Start the calculation with

In [3]:
%%bash
cd run_cBN_Borncharges/supercell
python3 -m excitingscripts.setup.excitingroot
python3 -m excitingscripts.execute.single
cd ../..

Once the calulcation is finished, you will find the result for the Born effective charges in the file **ZSTAR.OUT**. Its content looks like

```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (B  1) :      0.000000     0.000000     0.000000
        2.0047794515        0.0000088342        0.0000087245
        0.0000146861        2.0052228010        0.0000146861
        0.0000675616        0.0000675616        2.0019760711
# species  2 atom   1 (N  1) :      0.250000     0.250000     0.250000
       -2.0047794515       -0.0000088342       -0.0000087245
       -0.0000146861       -2.0052228010       -0.0000146861
       -0.0000675616       -0.0000675616       -2.0019760711
# Acoustic sum rule correction (add to each tensor above to get original value)
       -0.0000045591        0.0000017879        0.0000017451
        0.0000005652       -0.0000045701        0.0000005652
        0.0000005367        0.0000005367       -0.0000052148
```

For each atom in the unit cell, it gives the full charge tensor in units of the elementary charge. In cubic zincblende crystals, the charge tensor is diagonal and isotropic and can effectively be described by a scalar value. In this example, we obtain an effective charge of about **<span style="color:firebrick">±2.00</span>** for the boron and nitrogen atom, respectively.

In theory, the Born effective charge tensors obey the acoustic sum rule, *i.e.*, that the sum over all atoms is zero. We automatically impose the acoustic sum rule by subtracting the actual sum of all charge tensors divided by the total number of atoms from the charge tensor of each atom. The last block in the file gives the correction that was necessary to impose the acoustic sum rule. Ideally, it should be zero.

Note, that the actual values for the macroscopic polarization for each displacement can be found the files **POLARIZATION_*fileext*.OUT** in the directories **<span style="color:mediumblue">Q0000_0000_0000_S##_A###_P#</span>**, where ***fileext*** is the filename extension described in the tutorial **Lattice Dynamics of Diamond and Zincblende-Structure Crystals**.

<hr style="border:1px solid #DDD"> </hr>

<details>
<summary><strong><span style="color:firebrick">$\Rightarrow$ Show converged result</span></strong>
    
</summary> 
The result for a converged calculation using the parameters <code><span style="color:mediumblue">ngridk</span>=<span style="color:firebrick">"8 8 8"</span></code> and <code><span style="color:mediumblue">rgkmax</span>=<span style="color:firebrick">"7.0"</span></code> looks as follows:
    
```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (B  1) :      0.000000     0.000000     0.000000
        1.9269667472        0.0000018787        0.0000018750
       -0.0000037993        1.9265195605       -0.0000037993
        0.0000103316        0.0000103316        1.9291164244
# species  2 atom   1 (N  1) :      0.250000     0.250000     0.250000
       -1.9269667472       -0.0000018787       -0.0000018750
        0.0000037993       -1.9265195605        0.0000037993
       -0.0000103316       -0.0000103316       -1.9291164244
# Acoustic sum rule correction (add to each tensor above to get original value)
       -0.0000109212        0.0000007460        0.0000007524
        0.0000000671       -0.0000110205        0.0000000671
        0.0000000661        0.0000000661       -0.0000112378
```    

</details>
<hr style="border:1px solid #DDD"> </hr>  

<a id='3'></a>

### <span style="color:#15317E">3. Born Effective Charges from DFPT</span>

Alternatively, the derivatives can be evaluated using **density-functional perturbation theory** (**DFPT**) instead of finite differences. Create a new directory and copy the input file there.

In [4]:
%%bash
cd run_cBN_Borncharges
mkdir -p dfpt
cp supercell/input.xml dfpt/
cd ..

We only need to make a small change to the input file. We have to change <code><span style="color:blue">method</span></code> to <code><span style="color:firebrick">dfpt</span></code>.

```xml
<input>
 
   <title>cBN Born effective charges - DFPT</title>
   ...
   <phonons
      do="fromscratch"
      method="dfpt"
      ngridq="1 1 1"
      >
   </phonons>
 
</input>
```

In [7]:
# The following lines modify the input.xml using Python
from excitingtools import ExcitingInputXML

parsed_input = ExcitingInputXML.from_xml('./run_cBN_Borncharges/dfpt/input.xml')

parsed_input.phonons.method = "dfpt"

parsed_input.write('./run_cBN_Borncharges/dfpt/input.xml')

Run the calculation with

In [8]:
%%bash
cd run_cBN_Borncharges/dfpt
python3 -m excitingscripts.setup.excitingroot
python3 -m excitingscripts.execute.single
cd ../..

**ZSTAR.OUT** now looks like

```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (B  1) :      0.000000     0.000000     0.000000
        2.0052077990        0.0000000000        0.0000000000
        0.0000000000        2.0052077990        0.0000000000
        0.0000000000        0.0000000000        2.0052077990
# species  2 atom   1 (N  1) :      0.250000     0.250000     0.250000
       -2.0052077990        0.0000000000        0.0000000000
        0.0000000000       -2.0052077990        0.0000000000
        0.0000000000        0.0000000000       -2.0052077990
# Acoustic sum rule correction (add to each tensor above to get original value)
        0.0002667400        0.0000000000        0.0000000000
        0.0000000000        0.0002667400        0.0000000000
        0.0000000000        0.0000000000        0.0002667400
```

Note, that in the case of a DFPT calculation, the charge tensors are symmetrized using the crystal symmetries (in contrast to the supercell result).

<hr style="border:1px solid #DDD"> </hr>

<details>
<summary><strong><span style="color:firebrick">$\Rightarrow$ Show converged result</span></strong>
    
</summary> 

```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (B  1) :      0.000000     0.000000     0.000000
        1.9263699515        0.0000000000        0.0000000000
        0.0000000000        1.9263699515        0.0000000000
        0.0000000000        0.0000000000        1.9263699515
# species  2 atom   1 (N  1) :      0.250000     0.250000     0.250000
       -1.9263699515        0.0000000000        0.0000000000
        0.0000000000       -1.9263699515        0.0000000000
        0.0000000000        0.0000000000       -1.9263699515
# Acoustic sum rule correction (add to each tensor above to get original value)
        0.0002900905        0.0000000000        0.0000000000
        0.0000000000        0.0002900905        0.0000000000
        0.0000000000        0.0000000000        0.0002900905
```    

</details>
<hr style="border:1px solid #DDD"> </hr>  

<a id='4'></a>
**<span style="color:#15317E">Exercises</span>**
- The calculation presented here for cubic boron nitride can be repeated for silicon carbide. Use the PBE exchange-correlation functional and an equilibrium lattice parameter of 8.27884 Bohr.

<hr style="border:1px solid #DDD"> </hr>

<details>
<summary><strong><span style="color:firebrick">$\Rightarrow$ SiC: Result simple settings</span></strong>
    
</summary> 

```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (Si  1) :      0.000000     0.000000     0.000000
        2.6514921705        0.0000000000        0.0000000000
        0.0000000000        2.6514921705        0.0000000000
        0.0000000000        0.0000000000        2.6514921705
# species  2 atom   1 (C  1) :      0.250000     0.250000     0.250000
       -2.6514921705        0.0000000000        0.0000000000
        0.0000000000       -2.6514921705        0.0000000000
        0.0000000000        0.0000000000       -2.6514921705
# Acoustic sum rule correction (add to each tensor above to get original value)
        0.0011240375        0.0000000000        0.0000000000
        0.0000000000        0.0011240375        0.0000000000
        0.0000000000        0.0000000000        0.0011240375
```    

</details>
<hr style="border:1px solid #DDD"> </hr> 

<details>
<summary><strong><span style="color:firebrick">$\Rightarrow$ SiC: Result converged settings</span></strong>
    
</summary> 

```
# Born effective charge tensors for all atoms.
# Rows correspond to E-field direction.
# Columns correspond to atom displacement direction.
# Acoustic sum rule has been imposed.
#
# species  1 atom   1 (Si  1) :      0.000000     0.000000     0.000000
        2.7027482930        0.0000000000        0.0000000000
        0.0000000000        2.7027482930        0.0000000000
        0.0000000000        0.0000000000        2.7027482930
# species  2 atom   1 (C  1) :      0.250000     0.250000     0.250000
       -2.7027482930        0.0000000000        0.0000000000
        0.0000000000       -2.7027482930        0.0000000000
        0.0000000000        0.0000000000       -2.7027482930
# Acoustic sum rule correction (add to each tensor above to get original value)
        0.0014827560        0.0000000000        0.0000000000
        0.0000000000        0.0014827560        0.0000000000
        0.0000000000        0.0000000000        0.0014827560
```    

</details>
<hr style="border:1px solid #DDD"> </hr> 

- Which result do you expect for the Born effective charges in diamond? Examine the file **ZSTAR.OUT** in the directory **diamond_phonons/** in your home directory and explain the result.

<hr style="border:2px solid #DDD"> </hr> 